In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import seaborn as sns
from Bio import Entrez, Medline
import mermaid
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript
import subprocess, sys, os, ssl, certifi
import re
from pathlib import Path

In [2]:
import os 
from pathlib import Path
root = os.getcwd()
folders = {
    "systematic_review": f"{root}/systematic_review",
        "protocol": f"{root}/systematic_review/protocol",
            "prospero": f"{root}/systematic_review/protocol/prospero",
            "cochrane": f"{root}/systematic_review/protocol/cochrane",
        "search_strategy": f"{root}/systematic_review/search_strategy",
        "search": f"{root}/systematic_review/search",
        "deduplication": f"{root}/systematic_review/deduplication",
        "screening": f"{root}/systematic_review/screening",
            "title_abstract": f"{root}/systematic_review/screening/title_abstract_screening", 
            "pdf": f"{root}/systematic_review/screening/PDF",
            "full_text": f"{root}/systematic_review/screening/full_text_screening", 
    "data_collection": f"{root}/data_collection",
        "database": f"{root}/data_collection/database",
    "meta-analysis": f"{root}/meta-analysis",
    "manuscript": f"{root}/manuscript"
}

print(folders)
for x, y in folders.items():
    filename = f"{x}"
    path = Path(f"{y}")
    os.makedirs(path, exist_ok = True)
    globals()[filename] = path

{'systematic_review': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review', 'protocol': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/protocol', 'prospero': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/protocol/prospero', 'cochrane': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/protocol/cochrane', 'search_strategy': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/search_strategy', 'search': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/search', 'deduplication': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/deduplication', 'screening': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/screening', 'title_abstract': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/screening/title_abstract_screening', 'pdf': 'G:\\My Drive\\network_meta-analysis\\

In [29]:
import pandas as pd
import mermaid
import os

#input_file_name = f"{root}/systematic_review/deduplication/records.csv"

#filename = input('Enter file name: ')
#input_file_name = f"{search}/{filename}.csv"
#df = pd.read_csv(input_file_name, encoding = "utf-8") # A (records)
#cols_input = input('Enter the column for which to deduplicate based on: ')

# turn inputs into interactive widgets
upload = widgets.FileUpload()
output = widgets.Output()

with output:
    
    if upload.value:
        file = upload.value[0]
        content = file["content"]
        df = pd.read_csv(pd.io.common.BytesIO(content))
    
        nulls_mask = df["doi"].isnull().any(axis = 1)
        nulls = df[nulls_mask]
        non_nulls = df[~nulls]
    
cols = "doi"

folder = '_'.join(cols)
os.makedirs(f"{deduplication}/{folder}/", exist_ok = True)

output_file = f"{deduplication}/{folder}/{folder}_deduplicated"
output_file_name = f"{output_file}.csv"
output_for_recycle = f"{deduplication}/{folder}_deduplicated.csv"
prisma_file_name = f"{output_file}.mmd"

nulls_mask = df[cols].isnull().any(axis=1)
df_nulls = df[nulls_mask] # B
df_non_nulls = df[~nulls_mask] # C

duplicates_mask = df_non_nulls.duplicated(subset = cols, keep = False)
df_non_duplicates = df_non_nulls[~duplicates_mask] # D
df_duplicates = df_non_nulls[duplicates_mask] # E
#df_duplicates.groupby(cols, as_index=False).agg(agg_map)
df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: list(dict.fromkeys(s.dropna())) if s.name in ['subgroup', 'source'] else s.dropna().iloc[0] if len(s.dropna()) else pd.NA)
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: '; '.join(dict.fromkeys(s.dropna().astype(str).str.strip())) if s.name in ['subgroup', 'source'] else (s.dropna().astype(str).str.strip().iloc[0] if len(s.dropna().astype(str).str.strip()) else pd.NA))
df_removed = df_duplicates[~df_duplicates.index.isin(df_kept.index)]
#df_kept = df_merged.drop_duplicates(subset = cols, keep = 'first')
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: '; '.join(dict.fromkeys(s.dropna().astype(str).str.strip())) if s.name == 'source' and s.dropna().astype(str).str.strip().nunique() > 1 else (s.dropna().astype(str).str.strip().iloc[0] if len(s.dropna().astype(str).str.strip()) else pd.NA))
df_unique = df_non_nulls.drop_duplicates(subset = cols, keep = 'first') # df of unique
df_deduplicated = pd.concat([df_non_duplicates, df_kept, df_nulls], ignore_index=True) # df of unique + df of non-duplicates

results = {"records": len(df),  
"nulls": len(df_nulls), 
"non_nulls": len(df_non_nulls), 
"non_duplicates": len(df_non_duplicates), 
"duplicates": len(df_duplicates), 
"kept": len(df_kept),
"removed": len(df_duplicates) - len(df_kept), 
"unique": len(df_unique),
"deduplicated": len(df_deduplicated)
}

df_nulls.to_csv(output_file_name.replace('deduplicated','nulls'), index = False)
df_deduplicated.to_csv(output_file_name, index = False)
df_removed.to_csv(output_file_name.replace('deduplicated','duplicates_removed'), index = False)
df_deduplicated.to_csv(output_for_recycle, index = False)
print(results)

graph_text = f"""---
config:
theme: neutral
curve: stepBefore
---
graph TD;
A["`**records** (*n* = {results['records']})`"];
B["`null (*n* = {results['nulls']})`"];
C["`non-null (*n* = {results['non_nulls']})`"];
D["`non-duplicates (*n* = {results['non_duplicates']})`"];
E["`duplicates (*n* = {results['duplicates']})`"];
F["`duplicates kept (*n* = {results['kept']})`"];
G["`duplicates removed (*n* = {results['removed']})`"];
H["`unique (*n* = {results['unique']})`"];
I["`deduplicated (*n* = {results['deduplicated']})`"];

A --> B & C;
C --> D & E;
E --> F & G;
D & F --> H
B & H --> I"""

with open(prisma_file_name, "w") as f:
    f.write(graph_text)

!mmdc -i "{prisma_file_name}" -o "{output_file}"_light.svg
!mmdc -i "{prisma_file_name}" -o "{output_file}"_dark.svg -t dark -b transparent

display(upload)

NameError: name 'df' is not defined

In [7]:
columns = list(df.columns)
cols = widgets.ToggleButtons(options = columns, value = "doi")

cols = [c.strip() for c in cols_input.split(',')]
folder = '_'.join(cols)
os.makedirs(f"{deduplication}/{folder}/", exist_ok = True)

output_file = f"{deduplication}/{folder}/{folder}_deduplicated"
output_file_name = f"{output_file}.csv"
output_for_recycle = f"{deduplication}/{folder}_deduplicated.csv"
prisma_file_name = f"{output_file}.mmd"

nulls_mask = df[cols].isnull().any(axis=1)
df_nulls = df[nulls_mask] # B
df_non_nulls = df[~nulls_mask] # C

duplicates_mask = df_non_nulls.duplicated(subset = cols, keep = False)
df_non_duplicates = df_non_nulls[~duplicates_mask] # D
df_duplicates = df_non_nulls[duplicates_mask] # E
#df_duplicates.groupby(cols, as_index=False).agg(agg_map)
df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: list(dict.fromkeys(s.dropna())) if s.name in ['subgroup', 'source'] else s.dropna().iloc[0] if len(s.dropna()) else pd.NA)
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: '; '.join(dict.fromkeys(s.dropna().astype(str).str.strip())) if s.name in ['subgroup', 'source'] else (s.dropna().astype(str).str.strip().iloc[0] if len(s.dropna().astype(str).str.strip()) else pd.NA))
df_removed = df_duplicates[~df_duplicates.index.isin(df_kept.index)]
#df_kept = df_merged.drop_duplicates(subset = cols, keep = 'first')
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: '; '.join(dict.fromkeys(s.dropna().astype(str).str.strip())) if s.name == 'source' and s.dropna().astype(str).str.strip().nunique() > 1 else (s.dropna().astype(str).str.strip().iloc[0] if len(s.dropna().astype(str).str.strip()) else pd.NA))
df_unique = df_non_nulls.drop_duplicates(subset = cols, keep = 'first') # df of unique
df_deduplicated = pd.concat([df_non_duplicates, df_kept, df_nulls], ignore_index=True) # df of unique + df of non-duplicates

results = {"records": len(df),  
"nulls": len(df_nulls), 
"non_nulls": len(df_non_nulls), 
"non_duplicates": len(df_non_duplicates), 
"duplicates": len(df_duplicates), 
"kept": len(df_kept),
"removed": len(df_duplicates) - len(df_kept), 
"unique": len(df_unique),
"deduplicated": len(df_deduplicated)
}

df_nulls.to_csv(output_file_name.replace('deduplicated','nulls'), index = False)
df_deduplicated.to_csv(output_file_name, index = False)
df_removed.to_csv(output_file_name.replace('deduplicated','duplicates_removed'), index = False)
df_deduplicated.to_csv(output_for_recycle, index = False)
print(results)

graph_text = f"""---
config:
theme: neutral
curve: stepBefore
---
graph TD;
A["`**records** (*n* = {results['records']})`"];
B["`null (*n* = {results['nulls']})`"];
C["`non-null (*n* = {results['non_nulls']})`"];
D["`non-duplicates (*n* = {results['non_duplicates']})`"];
E["`duplicates (*n* = {results['duplicates']})`"];
F["`duplicates kept (*n* = {results['kept']})`"];
G["`duplicates removed (*n* = {results['removed']})`"];
H["`unique (*n* = {results['unique']})`"];
I["`deduplicated (*n* = {results['deduplicated']})`"];

A --> B & C;
C --> D & E;
E --> F & G;
D & F --> H
B & H --> I"""

with open(prisma_file_name, "w") as f:
    f.write(graph_text)

!mmdc -i "{prisma_file_name}" -o "{output_file}"_light.svg
!mmdc -i "{prisma_file_name}" -o "{output_file}"_dark.svg -t dark -b transparent

ValueError: Invalid file path or buffer object type: <class 'ipywidgets.widgets.widget_upload.FileUpload'>